# 🚀 E-Commerce Delivery Delay - CatBoost Cloud Training (TURBO)

### **Instructions:**
1. **Enable GPU**: `Runtime` -> `Change runtime type` -> **T4 GPU**.
2. **Manual Upload**: Click Folder Icon (📁) on left, drag/drop `features.csv`.
3. **Run All**: This version uses **30% sampling** for tuning, making it **10x faster**.

In [ ]:
import os
if os.path.exists("features.csv"):
    print(f"✅ Found data: {os.path.getsize('features.csv')/(1024*1024):.2f} MB")
else:
    print("❌ ERROR: Upload 'features.csv' to the sidebar first!")

In [ ]:
%pip install catboost optuna pandas

In [ ]:
import pandas as pd
import numpy as np
import optuna
import json
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score

# 1. Load Data
df = pd.read_csv("features.csv")
X = df.drop(columns=['order_id', 'is_late'], errors='ignore')
y = df['is_late'].astype(int)

cat_cols = ['customer_state', 'seller_state', 'product_category', 
            'primary_payment_type', 'purchase_month', 'purchase_day_of_week', 'purchase_hour']
cat_cols = [c for c in cat_cols if c in X.columns]
for col in cat_cols: X[col] = X[col].fillna("UNKNOWN").astype(str)

# 2. SAMPLE FOR TUNING (30%)
print("Sampling 30% of data for fast tuning...")
sample_idx = df.sample(frac=0.3, random_state=42).index
X_tune, y_tune = X.loc[sample_idx], y.loc[sample_idx]
print(f"Tuning on {len(X_tune)} rows, Final train will be on {len(X)} rows.")

In [ ]:
# --- 3. FAST OPTIMIZATION ---
def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 400, 1200),
        "depth": trial.suggest_int("depth", 4, 8), # Reduced max depth for speed
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.2), # Higher min LR for faster convergence
        "task_type": "GPU",
        "auto_class_weights": "Balanced",
        "verbose": False,
        "random_seed": 42
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for tr, val in skf.split(X_tune, y_tune):
        model = CatBoostClassifier(**params)
        model.fit(X_tune.iloc[tr], y_tune.iloc[tr], cat_features=cat_cols)
        probs = model.predict_proba(X_tune.iloc[val])[:, 1]
        scores.append(average_precision_score(y_tune.iloc[val], probs))
    return np.mean(scores)

print("Starting Fast Tuning (Estimated time: 10-15 mins)...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30) # Reduced to 30 trials for efficiency
print(f"\n🏆 Best PR-AUC on Sample: {study.best_value:.4f}")

In [ ]:
# --- 4. EXPORT & DOWNLOAD (FULL DATA) ---
print("Training final high-performance model on 100% of data...")
final_model = CatBoostClassifier(
    **study.best_params, 
    iterations=study.best_params['iterations'] + 200, # Add a few more trees for full data
    task_type="GPU", 
    auto_class_weights="Balanced", 
    verbose=100
)
final_model.fit(X, y, cat_features=cat_cols)
final_model.save_model("catboost_tuned.cbm")

with open("best_catboost_params.json", "w") as f: json.dump(study.best_params, f, indent=4)

from google.colab import files
!zip -j results.zip catboost_tuned.cbm *.json
files.download("results.zip")
print("✅ All done! Results downloaded.")